# exp054 NB1: Extract iNat + AnuraSet to BC2026-overlap Dataset (Kaggle CPU)

**Purpose**: Phase 2 — non-Aves coverage 補強用データ抽出

**Inputs**:
- `shadowdude/train-recordings` (iNat, 121 GB total)
- `bengtlueers/anuraset-v2-raw` (AnuraSet, 8.5 GB)

**Filter**: BC2026 234 species (主に non-Aves に効く)

**Output**:
- `/kaggle/working/extracted/audio/{source}/{primary_label}/*.ogg|wav`
- `metadata.csv` (filename, primary_label, source, scientific_name)
- Upload → `maekeso/birdclef2026-exp054-nonaves-extracted`

**Expected coverage**:
- iNat: ~24 Amphibia + 4 Mammalia + 3 Insecta + Aves overlap
- AnuraSet: 17 Amphibia
- Union: ~27 Amphibia + 4 Mammalia + 3 Insecta = **34 non-Aves species**

**Expected output size**: ~5-15 GB

**Use case**: Stage 1 v2 input、XC Part 1+2 と合わせて non-Aves backbone 強化


In [ ]:
# ============================================================
# Cell 1: Setup + species list
# ============================================================
import os, sys, json, time, re, shutil
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

OUT_DIR = Path("/kaggle/working/extracted")
OUT_DIR.mkdir(exist_ok=True, parents=True)
AUDIO_DIR = OUT_DIR / "audio"
AUDIO_DIR.mkdir(exist_ok=True)

# Safety caps
MAX_PER_SPECIES = 200  # extract 上限/species
STORAGE_CAP_GB = 18.0
print(f"Output: {OUT_DIR}")


In [ ]:
# ============================================================
# Cell 2: BC2026 species list (234)
# ============================================================
__BC2026_SPECIES = [
    ('Guyalna cuta', '1161364', 'Insecta'),
    ('Caiman yacare', '116570', 'Reptilia'),
    ('Leptodactylus luctator', '1176823', 'Amphibia'),
    ('Adenomera guarani', '1491113', 'Amphibia'),
    ('Lysapsus limellum', '1595929', 'Amphibia'),
    ('Equus caballus', '209233', 'Mammalia'),
    ('Leptodactylus syphax', '22930', 'Amphibia'),
    ('Leptodactylus mystacinus', '22956', 'Amphibia'),
    ('Leptodactylus podicipinus', '22961', 'Amphibia'),
    ('Leptodactylus elenae', '22967', 'Amphibia'),
    ('Leptodactylus fuscus', '22973', 'Amphibia'),
    ('Leptodactylus labyrinthicus', '22983', 'Amphibia'),
    ('Leptodactylus petersii', '22985', 'Amphibia'),
    ('Physalaemus centralis', '23150', 'Amphibia'),
    ('Physalaemus albifrons', '23154', 'Amphibia'),
    ('Physalaemus albonotatus', '23158', 'Amphibia'),
    ('Pseudopaludicola mystacalis', '23176', 'Amphibia'),
    ('Phyllomedusa sauvagii', '23724', 'Amphibia'),
    ('Scinax nasicus', '24279', 'Amphibia'),
    ('Scinax fuscovarius', '24285', 'Amphibia'),
    ('Scinax fuscomarginatus', '24287', 'Amphibia'),
    ('Scinax acuminatus', '24321', 'Amphibia'),
    ('Quesada gigas', '244024', 'Insecta'),
    ('Chiasmocleis mehelyi', '25073', 'Amphibia'),
    ('Elachistocleis bicolor', '25092', 'Amphibia'),
    ('Dermatonotus muelleri', '25214', 'Amphibia'),
    ('Physalaemus biligonigerus', '326272', 'Amphibia'),
    ('Panthera onca', '41970', 'Mammalia'),
    ('Alouatta caraya', '43435', 'Mammalia'),
    ('Canis familiaris', '47144', 'Mammalia'),
    ('Insect son01', '47158son01', 'Insecta'),
    ('Insect son02', '47158son02', 'Insecta'),
    ('Insect son03', '47158son03', 'Insecta'),
    ('Insect son04', '47158son04', 'Insecta'),
    ('Insect son05', '47158son05', 'Insecta'),
    ('Insect son06', '47158son06', 'Insecta'),
    ('Insect son07', '47158son07', 'Insecta'),
    ('Insect son08', '47158son08', 'Insecta'),
    ('Insect son09', '47158son09', 'Insecta'),
    ('Insect son10', '47158son10', 'Insecta'),
    ('Insect son11', '47158son11', 'Insecta'),
    ('Insect son12', '47158son12', 'Insecta'),
    ('Insect son13', '47158son13', 'Insecta'),
    ('Insect son14', '47158son14', 'Insecta'),
    ('Insect son15', '47158son15', 'Insecta'),
    ('Insect son16', '47158son16', 'Insecta'),
    ('Insect son17', '47158son17', 'Insecta'),
    ('Insect son18', '47158son18', 'Insecta'),
    ('Insect son19', '47158son19', 'Insecta'),
    ('Insect son20', '47158son20', 'Insecta'),
    ('Insect son21', '47158son21', 'Insecta'),
    ('Insect son22', '47158son22', 'Insecta'),
    ('Insect son23', '47158son23', 'Insecta'),
    ('Insect son24', '47158son24', 'Insecta'),
    ('Insect son25', '47158son25', 'Insecta'),
    ('Physalaemus nattereri', '476521', 'Amphibia'),
    ('Sapajus cay', '516975', 'Mammalia'),
    ('Pithecopus azureus', '517063', 'Amphibia'),
    ('Boana lundii', '555123', 'Amphibia'),
    ('Boana punctata', '555145', 'Amphibia'),
    ('Boana raniceps', '555146', 'Amphibia'),
    ('Ameerega picta', '64898', 'Amphibia'),
    ('Dendropsophus minutus', '65377', 'Amphibia'),
    ('Dendropsophus nanus', '65380', 'Amphibia'),
    ('Pseudis platensis', '66971', 'Amphibia'),
    ('Rhinella diptycha', '67107', 'Amphibia'),
    ('Trachycephalus typhonius', '67252', 'Amphibia'),
    ('Leptodactylus macrosternum', '70711', 'Amphibia'),
    ('Plecturocebus pallescens', '738183', 'Mammalia'),
    ('Bos taurus', '74113', 'Mammalia'),
    ('Mico melanurus', '74580', 'Mammalia'),
    ('Prionacris erosa', '760266', 'Insecta'),
    ('Hylophilus pectoralis', 'ashgre1', 'Aves'),
    ('Mustelirallus albicollis', 'astcra1', 'Aves'),
    ('Crax fasciolata', 'bafcur1', 'Aves'),
    ('Micrastur ruficollis', 'baffal1', 'Aves'),
    ('Coereba flaveola', 'banana', 'Aves'),
    ('Thamnophilus doliatus', 'barant1', 'Aves'),
    ('Procnias nudicollis', 'batbel1', 'Aves'),
    ('Ara ararauna', 'baymac', 'Aves'),
    ('Dendrocygna autumnalis', 'bbwduc', 'Aves'),
    ('Microspingus melanoleucus', 'bcwfin2', 'Aves'),
    ('Donacobius atricapilla', 'bkcdon', 'Aves'),
    ('Aratinga nenday', 'bkhpar', 'Aves'),
    ('Busarellus nigricollis', 'blchaw1', 'Aves'),
    ('Spizaetus tyrannus', 'blheag1', 'Aves'),
    ('Tityra cayana', 'blttit1', 'Aves'),
    ('Myiarchus tyrannulus', 'bncfly', 'Aves'),
    ('Megarynchus pitangua', 'bobfly1', 'Aves'),
    ('Progne tapera', 'brcmar1', 'Aves'),
    ('Tyto furcata', 'brnowl', 'Aves'),
    ('Momotus momota', 'bucmot4', 'Aves'),
    ('Thectocercus acuticaudatus', 'bucpar', 'Aves'),
    ('Amazona aestiva', 'bufpar', 'Aves'),
    ('Theristicus caudatus', 'bunibi1', 'Aves'),
    ('Athene cunicularia', 'burowl', 'Aves'),
    ('Colaptes campestris', 'camfli1', 'Aves'),
    ('Ortalis canicollis', 'chacha1', 'Aves'),
    ('Mimus saturninus', 'chbmoc1', 'Aves'),
    ('Gnorimopsar chopi', 'chobla1', 'Aves'),
    ('Conirostrum speciosum', 'chvcon1', 'Aves'),
    ('Synallaxis hypospodia', 'cibspi1', 'Aves'),
    ('Micrastur semitorquatus', 'coffal1', 'Aves'),
    ('Nyctidromus albicollis', 'compau', 'Aves'),
    ('Nyctibius griseus', 'compot1', 'Aves'),
    ('Turdus amaurochalinus', 'crbthr1', 'Aves'),
    ('Pachyramphus validus', 'crebec1', 'Aves'),
    ('Taoniscus nanus', 'dwatin1', 'Aves'),
    ('Icterus pyrrhopterus', 'epaori4', 'Aves'),
    ('Lathrotriccus euleri', 'eulfly1', 'Aves'),
    ('Cantorchilus guarayanus', 'fabwre1', 'Aves'),
    ('Glaucidium brasilianum', 'fepowl', 'Aves'),
    ('Machaeropterus pyrocephalus', 'ficman1', 'Aves'),
    ('Myiothlypis flaveola', 'flawar1', 'Aves'),
    ('Tyrannus savana', 'fotfly', 'Aves'),
    ('Cnemotriccus fuscatus', 'fusfly1', 'Aves'),
    ('Hylocharis chrysura', 'gilhum1', 'Aves'),
    ('Aramides ypecaha', 'giwrai1', 'Aves'),
    ('Chionomesa fimbriata', 'glteme1', 'Aves'),
    ('Saltator coerulescens', 'grasal3', 'Aves'),
    ('Crotophaga major', 'greani1', 'Aves'),
    ('Taraba major', 'greant1', 'Aves'),
    ('Myiopagis viridicata', 'greela', 'Aves'),
    ('Pitangus sulphuratus', 'grekis', 'Aves'),
    ('Nyctibius grandis', 'grepot1', 'Aves'),
    ('Phacellodomus ruber', 'gretho2', 'Aves'),
    ('Tringa melanoleuca', 'greyel', 'Aves'),
    ('Leptotila rufaxilla', 'grfdov1', 'Aves'),
    ('Eucometis penicillata', 'grhtan1', 'Aves'),
    ('Aramides cajaneus', 'gycwor1', 'Aves'),
    ('Anhima cornuta', 'horscr1', 'Aves'),
    ('Passer domesticus', 'houspa', 'Aves'),
    ('Anodorhynchus hyacinthinus', 'hyamac1', 'Aves'),
    ('Elaenia spectabilis', 'larela1', 'Aves'),
    ('Elaenia chiriquensis', 'lesela1', 'Aves'),
    ('Emberizoides ypiranganus', 'lesgrf1', 'Aves'),
    ('Aramus guarauna', 'limpki', 'Aves'),
    ('Dryocopus lineatus', 'linwoo1', 'Aves'),
    ('Coccycua minuta', 'litcuc2', 'Aves'),
    ('Setopagis parvula', 'litnig1', 'Aves'),
    ('Pyrrhura frontalis', 'mabpar', 'Aves'),
    ('Cercomacra melanaria', 'magant1', 'Aves'),
    ('Cissopis leverianus', 'magtan2', 'Aves'),
    ('Polioptila dumicola', 'masgna1', 'Aves'),
    ('Chordeiles nacunda', 'nacnig1', 'Aves'),
    ('Rufirallus schomburgkii', 'ocecra1', 'Aves'),
    ('Sittasomus griseicapillus', 'oliwoo1', 'Aves'),
    ('Icterus croconotus', 'orbtro3', 'Aves'),
    ('Amazona amazonica', 'orwpar', 'Aves'),
    ('Pandion haliaetus', 'osprey', 'Aves'),
    ('Synallaxis albescens', 'pabspi1', 'Aves'),
    ('Furnarius leucopus', 'palhor3', 'Aves'),
    ('Thraupis palmarum', 'paltan1', 'Aves'),
    ('Dromococcyx phasianellus', 'phecuc1', 'Aves'),
    ('Patagioenas picazuro', 'picpig2', 'Aves'),
    ('Legatus leucophaius', 'pirfly1', 'Aves'),
    ('Thamnophilus pelzelni', 'plasla1', 'Aves'),
    ('Inezia inornata', 'platyr1', 'Aves'),
    ('Cyanocorax chrysops', 'plcjay1', 'Aves'),
    ('Theristicus caerulescens', 'pluibi1', 'Aves'),
    ('Cyanocorax cyanomelas', 'purjay1', 'Aves'),
    ('Hemitriccus margaritaceiventer', 'pvttyr1', 'Aves'),
    ('Ara chloropterus', 'ragmac1', 'Aves'),
    ('Campylorhamphus trochilirostris', 'rebscy1', 'Aves'),
    ('Coryphospingus cucullatus', 'recfin1', 'Aves'),
    ('Gallus gallus', 'redjun', 'Aves'),
    ('Cariama cristata', 'relser1', 'Aves'),
    ('Megaceryle torquata', 'rinkin1', 'Aves'),
    ('Myiothlypis rivularis', 'rivwar1', 'Aves'),
    ('Rupornis magnirostris', 'roahaw', 'Aves'),
    ('Turdus rufiventris', 'rubthr1', 'Aves'),
    ('Pseudoseisura unirufa', 'rufcac2', 'Aves'),
    ('Casiornis rufus', 'rufcas2', 'Aves'),
    ('Conopophaga lineata', 'rufgna3', 'Aves'),
    ('Furnarius rufus', 'rufhor2', 'Aves'),
    ('Antrostomus rufus', 'rufnig1', 'Aves'),
    ('Phacellodomus rufifrons', 'ruftho1', 'Aves'),
    ('Poecilotriccus latirostris', 'ruftof1', 'Aves'),
    ('Myiozetetes cayanensis', 'rumfly1', 'Aves'),
    ('Tigrisoma lineatum', 'ruther1', 'Aves'),
    ('Galbula ruficauda', 'rutjac1', 'Aves'),
    ('Arremon flavirostris', 'sabspa1', 'Aves'),
    ('Sicalis flaveola', 'saffin', 'Aves'),
    ('Thraupis sayaca', 'saytan1', 'Aves'),
    ('Columbina squammata', 'scadov1', 'Aves'),
    ('Pionus maximiliani', 'schpar1', 'Aves'),
    ('Phaethornis eurynome', 'scther1', 'Aves'),
    ('Myiarchus ferox', 'shcfly1', 'Aves'),
    ('Accipiter striatus', 'shshaw', 'Aves'),
    ('Lurocalis semitorquatus', 'shtnig1', 'Aves'),
    ('Ramphocelus carbo', 'sibtan2', 'Aves'),
    ('Crotophaga ani', 'smbani', 'Aves'),
    ('Crypturellus parvirostris', 'smbtin1', 'Aves'),
    ('Cacicus solitarius', 'sobcac1', 'Aves'),
    ('Camptostoma obsoletum', 'sobtyr1', 'Aves'),
    ('Myiozetetes similis', 'socfly1', 'Aves'),
    ('Synallaxis frontalis', 'sofspi1', 'Aves'),
    ('Corythopis delalandi', 'souant1', 'Aves'),
    ('Vanellus chilensis', 'soulap1', 'Aves'),
    ('Chauna torquata', 'souscr1', 'Aves'),
    ('Hypoedaleus guttatus', 'spbant3', 'Aves'),
    ('Synallaxis spixi', 'spispi1', 'Aves'),
    ('Antiurus maculicaudus', 'sptnig1', 'Aves'),
    ('Piaya cayana', 'squcuc1', 'Aves'),
    ('Dendroplex picus', 'stbwoo2', 'Aves'),
    ('Tapera naevia', 'strcuc1', 'Aves'),
    ('Butorides striata', 'strher2', 'Aves'),
    ('Asio clamator', 'strowl1', 'Aves'),
    ('Eupetomena macroura', 'swthum1', 'Aves'),
    ('Chiroxiphia caudata', 'swtman1', 'Aves'),
    ('Crypturellus tataupa', 'tattin1', 'Aves'),
    ('Campylorhynchus turdinus', 'thlwre1', 'Aves'),
    ('Ramphastos toco', 'toctou1', 'Aves'),
    ('Tyrannus melancholicus', 'trokin', 'Aves'),
    ('Megascops choliba', 'trsowl', 'Aves'),
    ('Crypturellus undulatus', 'undtin1', 'Aves'),
    ('Thamnophilus caerulescens', 'varant1', 'Aves'),
    ('Jacana jacana', 'watjac1', 'Aves'),
    ('Pyriglena maura', 'wesfie1', 'Aves'),
    ('Dendrocygna viduata', 'wfwduc1', 'Aves'),
    ('Biatas nigropectus', 'whbant2', 'Aves'),
    ('Myiothlypis leucoblephara', 'whbwar2', 'Aves'),
    ('Melanerpes candidus', 'whiwoo1', 'Aves'),
    ('Synallaxis albilora', 'whlspi1', 'Aves'),
    ('Cyanocorax cyanopogon', 'whnjay1', 'Aves'),
    ('Leptotila verreauxi', 'whtdov', 'Aves'),
    ('Picumnus albosquamatus', 'whwpic1', 'Aves'),
    ('Caracara plancus', 'y00678', 'Aves'),
    ('Paroaria capitata', 'yebcar', 'Aves'),
    ('Elaenia flavogaster', 'yebela1', 'Aves'),
    ('Primolius auricollis', 'yecmac', 'Aves'),
    ('Brotogeris chiriri', 'yecpar', 'Aves'),
    ('Daptrius chimachima', 'yehcar1', 'Aves'),
    ('Tolmomyias sulphurescens', 'yeofly1', 'Aves'),
]
species_df = pd.DataFrame(__BC2026_SPECIES, columns=["scientific_name", "primary_label", "class_name"])
SCI_LC_TO_LABEL = {str(s).lower(): l for s, l in zip(species_df['scientific_name'], species_df['primary_label'])}
LABEL_TO_CLASS = dict(zip(species_df['primary_label'], species_df['class_name']))
LABEL_SET = set(species_df['primary_label'])
SCI_LC_SET = set(SCI_LC_TO_LABEL.keys())
print(f"BC2026: {len(species_df)} species")
print(f"  classes: {species_df['class_name'].value_counts().to_dict()}")


In [ ]:
# ============================================================
# Cell 3: Helpers
# ============================================================
def safe_dir(name):
    return re.sub(r"[^A-Za-z0-9_-]", "_", str(name))

def check_storage_gb():
    try:
        return sum(f.stat().st_size for f in AUDIO_DIR.rglob("*") if f.is_file()) / 1e9
    except Exception:
        return 0.0

all_metadata = []


In [ ]:
# ============================================================
# Cell 4: Extract AnuraSet
# ============================================================
ANURA_CANDIDATES = [
    Path("/kaggle/input/datasets/bengtlueers/anuraset-v2-raw"),
    Path("/kaggle/input/anuraset-v2-raw"),
    Path("/kaggle/input/bengtlueers-anuraset-v2-raw"),
]
anura_root = next((p for p in ANURA_CANDIDATES if p.exists()), None)

if anura_root is None:
    print(f"AnuraSet not mounted - skip. Tried: {ANURA_CANDIDATES}")
else:
    print(f"AnuraSet root: {anura_root}")
    # Find metadata
    meta_candidates = list(anura_root.rglob("metadata*.csv")) + list(anura_root.rglob("*labels*.csv"))
    print(f"  CSV candidates: {[str(c.relative_to(anura_root)) for c in meta_candidates[:5]]}")

    if meta_candidates:
        meta_path = meta_candidates[0]
        df = pd.read_csv(meta_path)
        print(f"  metadata: {len(df)} rows, columns: {df.columns.tolist()[:10]}")

        # Try to find scientific_name column
        sci_col_candidates = ["scientific_name", "species", "species_name", "label", "taxon"]
        sci_col = next((c for c in sci_col_candidates if c in df.columns), None)
        print(f"  sci_col: {sci_col}")

        # Try to find filename column
        name_col_candidates = ["filename", "file", "audio", "path", "wav_path", "audio_path"]
        name_col = next((c for c in name_col_candidates if c in df.columns), None)
        print(f"  name_col: {name_col}")

        if sci_col and name_col:
            # Filter to BC2026 species
            df["_sci_lc"] = df[sci_col].astype(str).str.lower()
            df["_in_bc26"] = df["_sci_lc"].isin(SCI_LC_SET)
            df_match = df[df["_in_bc26"]].reset_index(drop=True)
            print(f"  matched: {len(df_match)} / {len(df)}")
            if len(df_match) > 0:
                print(f"  matched species: {df_match[sci_col].unique().tolist()}")

                # Copy files (with per-species cap)
                df_match = df_match.groupby("_sci_lc", group_keys=False).head(MAX_PER_SPECIES).reset_index(drop=True)
                print(f"  after cap {MAX_PER_SPECIES}: {len(df_match)}")

                n_copied = 0
                for _, r in tqdm(df_match.iterrows(), total=len(df_match), desc="AnuraSet copy"):
                    sci = r[sci_col]
                    label = SCI_LC_TO_LABEL.get(str(sci).lower())
                    if not label: continue

                    rel = str(r[name_col])
                    src = anura_root / rel
                    if not src.exists():
                        # Try anura subdir structure
                        for candidate in anura_root.rglob(Path(rel).name):
                            src = candidate
                            break
                    if not src.exists(): continue

                    dst_dir = AUDIO_DIR / "anuraset" / safe_dir(label)
                    dst_dir.mkdir(parents=True, exist_ok=True)
                    dst = dst_dir / src.name
                    if dst.exists(): continue

                    try:
                        shutil.copy2(src, dst)
                        n_copied += 1
                        all_metadata.append({
                            "filename": str(dst.relative_to(OUT_DIR)),
                            "primary_label": label,
                            "scientific_name": str(sci),
                            "source": "anuraset",
                            "class_name": LABEL_TO_CLASS.get(label, ""),
                            "file_size_mb": dst.stat().st_size / 1e6,
                        })
                    except Exception as e:
                        pass

                    if check_storage_gb() > STORAGE_CAP_GB:
                        print(f"⚠ storage cap reached, stopping")
                        break
                print(f"  AnuraSet copied: {n_copied} files")
            else:
                print(f"  no BC2026 species matched in AnuraSet")
        else:
            print(f"  could not identify sci_name / filename columns")
            print(f"  columns full: {df.columns.tolist()}")


In [ ]:
# ============================================================
# Cell 5: Extract iNat (large 121 GB, careful filter)
# ============================================================
INAT_CANDIDATES = [
    Path("/kaggle/input/datasets/shadowdude/train-recordings"),
    Path("/kaggle/input/train-recordings"),
    Path("/kaggle/input/shadowdude-train-recordings"),
]
inat_root = next((p for p in INAT_CANDIDATES if p.exists()), None)

if inat_root is None:
    print(f"iNat not mounted - skip. Tried: {INAT_CANDIDATES}")
else:
    print(f"iNat root: {inat_root}")
    # iNat structure auto-discovery
    csv_candidates = list(inat_root.rglob("*.csv"))[:10]
    print(f"  CSV candidates: {[str(c.relative_to(inat_root)) for c in csv_candidates]}")

    if csv_candidates:
        meta_path = csv_candidates[0]  # first CSV
        df = pd.read_csv(meta_path)
        print(f"  metadata: {len(df)} rows, columns: {df.columns.tolist()[:10]}")

        sci_col_candidates = ["scientific_name", "species_name", "species", "name", "taxon"]
        sci_col = next((c for c in sci_col_candidates if c in df.columns), None)
        name_col_candidates = ["filename", "file", "audio_id", "path", "filepath", "file_name"]
        name_col = next((c for c in name_col_candidates if c in df.columns), None)
        print(f"  sci_col: {sci_col}, name_col: {name_col}")

        if sci_col and name_col:
            df["_sci_lc"] = df[sci_col].astype(str).str.lower()
            df["_in_bc26"] = df["_sci_lc"].isin(SCI_LC_SET)
            df_match = df[df["_in_bc26"]].reset_index(drop=True)
            print(f"  matched: {len(df_match)} / {len(df)}")
            if len(df_match) > 0:
                print(f"  matched species sample: {df_match[sci_col].unique()[:10].tolist()}")
                print(f"  matched class breakdown:")
                for _, r in df_match.head(50).iterrows():
                    pass

                # per-species cap
                df_match = df_match.groupby("_sci_lc", group_keys=False).head(MAX_PER_SPECIES).reset_index(drop=True)
                print(f"  after cap {MAX_PER_SPECIES}: {len(df_match)}")

                n_copied = 0
                for _, r in tqdm(df_match.iterrows(), total=len(df_match), desc="iNat copy"):
                    sci = r[sci_col]
                    label = SCI_LC_TO_LABEL.get(str(sci).lower())
                    if not label: continue

                    rel = str(r[name_col])
                    src = inat_root / rel
                    if not src.exists():
                        # Try discovery
                        for c in inat_root.rglob(Path(rel).name):
                            src = c
                            break
                    if not src.exists(): continue

                    dst_dir = AUDIO_DIR / "inat" / safe_dir(label)
                    dst_dir.mkdir(parents=True, exist_ok=True)
                    dst = dst_dir / src.name
                    if dst.exists(): continue

                    try:
                        shutil.copy2(src, dst)
                        n_copied += 1
                        all_metadata.append({
                            "filename": str(dst.relative_to(OUT_DIR)),
                            "primary_label": label,
                            "scientific_name": str(sci),
                            "source": "inat",
                            "class_name": LABEL_TO_CLASS.get(label, ""),
                            "file_size_mb": dst.stat().st_size / 1e6,
                        })
                    except Exception:
                        pass

                    if check_storage_gb() > STORAGE_CAP_GB:
                        print(f"⚠ storage cap reached, stopping")
                        break
                print(f"  iNat copied: {n_copied} files")
            else:
                print(f"  no BC2026 species matched in iNat")
        else:
            print(f"  could not identify sci_name / filename columns")
            print(f"  columns: {df.columns.tolist()}")


In [ ]:
# ============================================================
# Cell 6: Summary + save metadata
# ============================================================
if all_metadata:
    final_df = pd.DataFrame(all_metadata)
    print(f"Total extracted: {len(final_df)} files")
    print(f"  by source: {final_df['source'].value_counts().to_dict()}")
    print(f"  by class: {final_df['class_name'].value_counts().to_dict()}")
    print(f"  unique species: {final_df['primary_label'].nunique()} / 234")
    print(f"  total size: {final_df['file_size_mb'].sum()/1024:.2f} GB")

    final_df.to_csv(OUT_DIR / "metadata.csv", index=False)
    print(f"\nSaved: {OUT_DIR / 'metadata.csv'}")

    # Per-species
    sp = final_df.groupby(["primary_label", "class_name"]).agg(
        n_files=("filename", "count"),
        n_sources=("source", "nunique"),
        total_mb=("file_size_mb", "sum"),
    ).reset_index().sort_values("n_files", ascending=False)
    sp.to_csv(OUT_DIR / "per_species.csv", index=False)
    print(f"\nTop 10 species:")
    print(sp.head(10).to_string(index=False))
else:
    print("No metadata accumulated")


In [ ]:
# ============================================================
# Cell 7: Upload as Kaggle Dataset
# ============================================================
import json, shutil
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()

USER = "maekeso"
SLUG = "birdclef2026-exp054-nonaves-extracted"
TITLE = "BirdCLEF2026 exp054 Non-Aves Extracted (iNat + AnuraSet)"

DRY_RUN = False  # set False to upload

if not DRY_RUN:
    meta = {
        "title": TITLE,
        "id": f"{USER}/{SLUG}",
        "licenses": [{"name": "other"}],
    }
    (OUT_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))

    try:
        api.dataset_create_version(folder=str(OUT_DIR),
                                    version_notes="Phase 2 non-Aves extracted",
                                    dir_mode="zip", quiet=False)
        print("OK new version uploaded")
    except Exception:
        try:
            api.dataset_create_new(folder=str(OUT_DIR), public=False,
                                    dir_mode="zip", quiet=False)
            print("OK new dataset created")
        except Exception as e:
            print(f"upload err: {str(e)[:300]}")
    print(f"URL: https://www.kaggle.com/datasets/{USER}/{SLUG}")
else:
    print(f"DRY_RUN={DRY_RUN}, skip upload")
